In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/olist.db')

print("✅ Connected to database")

✅ Connected to database


In [2]:
query1 = """
SELECT 
    strftime('%Y-%m', o.order_purchase_timestamp) as month,
    COUNT(DISTINCT o.order_id) as total_orders,
    SUM(oi.price + oi.freight_value) as total_revenue,
    ROUND(AVG(oi.price + oi.freight_value), 2) as avg_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY 1;
"""

df1 = pd.read_sql(query1, conn)
print("REVENUE TREND BY MONTH:")
print("="*50)
print(df1.to_string())

REVENUE TREND BY MONTH:
      month  total_orders  total_revenue  avg_order_value
0   2016-09             1         143.46            47.82
1   2016-10           265       46490.66           148.53
2   2016-12             1          19.62            19.62
3   2017-01           750      127482.37           139.63
4   2017-02          1653      271239.32           145.98
5   2017-03          2546      414330.95           143.02
6   2017-04          2303      390812.40           152.13
7   2017-05          3546      566851.40           141.57
8   2017-06          3135      490050.37           140.46
9   2017-07          3872      566299.08           128.24
10  2017-08          4193      645832.36           134.63
11  2017-09          4150      701077.49           148.00
12  2017-10          4478      751117.01           144.06
13  2017-11          7289     1153364.20           136.09
14  2017-12          5513      843078.29           136.27
15  2018-01          7069     1077887.46        

In [3]:
query2 = """
SELECT 
    p.product_category_name_english as category,
    COUNT(DISTINCT oi.order_id) as total_orders,
    SUM(oi.price) as total_revenue,
    ROUND(AVG(oi.price), 2) as avg_price
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY total_revenue DESC
LIMIT 10;
"""

df2 = pd.read_sql(query2, conn)
print("TOP 10 CATEGORIES BY REVENUE:")
print("="*50)
print(df2.to_string())

TOP 10 CATEGORIES BY REVENUE:
                category  total_orders  total_revenue  avg_price
0          health_beauty          8647     1233131.72     130.28
1          watches_gifts          5495     1166176.98     199.04
2         bed_bath_table          9272     1023434.76      93.44
3         sports_leisure          7530      954852.55     113.25
4  computers_accessories          6530      888724.61     116.26
5        furniture_decor          6307      711927.69      87.25
6             housewares          5743      615628.69      90.60
7             cool_stuff          3559      610204.10     164.12
8                   auto          3810      578966.65     139.85
9                   toys          3804      471286.48     116.94


In [4]:
query3 = """
SELECT 
    c.customer_state,
    COUNT(DISTINCT o.order_id) as total_orders,
    ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 2) as avg_delivery_days,
    ROUND(SUM(CASE 
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 
        ELSE 0 
    END) * 100.0 / COUNT(*), 2) as late_delivery_pct
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY 1
ORDER BY avg_delivery_days;
"""

df3 = pd.read_sql(query3, conn)
print("DELIVERY PERFORMANCE BY STATE:")
print("="*50)
print(df3.to_string())

DELIVERY PERFORMANCE BY STATE:
   customer_state  total_orders  avg_delivery_days  late_delivery_pct
0              SP         40494               8.76               5.89
1              PR          4923              11.99               5.00
2              MG         11354              12.01               5.61
3              DF          2080              12.97               7.07
4              SC          3546              14.95               9.76
5              RS          5344              15.30               7.15
6              RJ         12350              15.31              13.47
7              GO          1957              15.61               8.18
8              MS           701              15.62              11.55
9              ES          1995              15.79              12.23
10             TO           274              17.66              12.77
11             MT           886              18.06               6.77
12             PE          1593              18.45         

In [5]:
query4 = """
SELECT 
    payment_type,
    COUNT(DISTINCT order_id) as total_transactions,
    ROUND(AVG(payment_value), 2) as avg_payment_value,
    SUM(payment_value) as total_payment_value,
    ROUND(AVG(payment_installments), 1) as avg_installments
FROM order_payments
GROUP BY 1
ORDER BY total_payment_value DESC;
"""

df4 = pd.read_sql(query4, conn)
print("PAYMENT METHOD ANALYSIS:")
print("="*50)
print(df4.to_string())

PAYMENT METHOD ANALYSIS:
  payment_type  total_transactions  avg_payment_value  total_payment_value  avg_installments
0  credit_card               76505             163.32         1.254208e+07               3.5
1       boleto               19784             145.03         2.869361e+06               1.0
2      voucher                3866              65.70         3.794369e+05               1.0
3   debit_card                1528             142.57         2.179898e+05               1.0
4  not_defined                   3               0.00         0.000000e+00               1.0


In [6]:
query5 = """
SELECT 
    p.product_category_name_english as category,
    ROUND(AVG(r.review_score), 2) as avg_review_score,
    COUNT(DISTINCT r.review_id) as total_reviews,
    ROUND(SUM(CASE WHEN r.review_score >= 4 THEN 1 END) * 100.0 / COUNT(*), 2) as satisfaction_rate
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN orders o ON oi.order_id = o.order_id
JOIN order_reviews r ON o.order_id = r.order_id
WHERE r.review_score IS NOT NULL
GROUP BY 1
HAVING COUNT(*) >= 50
ORDER BY avg_review_score DESC
LIMIT 10;
"""

df5 = pd.read_sql(query5, conn)
print("TOP CATEGORIES BY REVIEW SCORE:")
print("="*50)
print(df5.to_string())

TOP CATEGORIES BY REVIEW SCORE:
                                category  avg_review_score  total_reviews  satisfaction_rate
0                 books_general_interest              4.45            508              87.80
1                costruction_tools_tools              4.44             94              90.91
2                         books_imported              4.40             53              85.00
3                        books_technical              4.37            259              85.34
4                    luggage_accessories              4.32           1030              83.73
5                             food_drink              4.32            226              81.72
6  small_appliances_home_oven_and_coffee              4.30             75              84.21
7                          fashion_shoes              4.23            239              80.84
8                                   food              4.22            445              81.82
9                             cine_pho

In [7]:
query6 = """
WITH customer_rfm AS (
    SELECT 
        c.customer_unique_id,
        c.customer_state,
        MAX(o.order_purchase_timestamp) as last_purchase,
        COUNT(DISTINCT o.order_id) as frequency,
        SUM(oi.price + oi.freight_value) as monetary,
        CAST(julianday('now') - julianday(MAX(o.order_purchase_timestamp)) AS INTEGER) as recency_days
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1, 2
)
SELECT 
    customer_unique_id,
    customer_state,
    recency_days,
    frequency,
    ROUND(monetary, 2) as monetary,
    CASE 
        WHEN recency_days <= 30 THEN 'Active'
        WHEN recency_days <= 90 THEN 'Recent'
        WHEN recency_days <= 180 THEN 'At Risk'
        ELSE 'Lost'
    END as recency_segment,
    CASE 
        WHEN frequency >= 5 THEN 'High'
        WHEN frequency >= 2 THEN 'Medium'
        ELSE 'Low'
    END as frequency_segment
FROM customer_rfm
LIMIT 20;
"""

df6 = pd.read_sql(query6, conn)
print("RFM CUSTOMER SEGMENTATION (SAMPLE):")
print("="*50)
print(df6.to_string())

RFM CUSTOMER SEGMENTATION (SAMPLE):
                  customer_unique_id customer_state  recency_days  frequency  monetary recency_segment frequency_segment
0   0000366f3b9a7992bf8c76cfdf3221e2             SP          2881          1    141.90            Lost               Low
1   0000b849f77a49e4a4ce2b2a4ca5be3f             SP          2884          1     27.19            Lost               Low
2   0000f46a3911fa3c0805444483337064             SC          3307          1     86.22            Lost               Low
3   0000f6ccb0745a6a4b88665a16c9f078             PA          3091          1     43.62            Lost               Low
4   0004aac84e0df4da2b147fca70cf8255             SP          3058          1    196.89            Lost               Low
5   0004bd2a26a76fe21f786e4fbd80607f             SP          2916          1    166.98            Lost               Low
6   00050ab1314c0e55a6ca13cf7181fecf             SP          2901          1     35.38            Lost               

In [8]:
df1.to_csv('images/revenue_trend_data.csv', index=False)
df2.to_csv('images/top_categories_data.csv', index=False)
df3.to_csv('images/delivery_performance_data.csv', index=False)
df4.to_csv('images/payment_analysis_data.csv', index=False)
df5.to_csv('images/review_score_data.csv', index=False)

print("✅ Semua query results telah disimpan ke folder images/")

conn.close()
print("✅ Database connection closed")

✅ Semua query results telah disimpan ke folder images/
✅ Database connection closed
